# 116. Voronoi分割: query/passageプレフィックスの影響

## 背景
E5モデルは `query: ` と `passage: ` のプレフィックスで**異なる空間にマッピング**する（NB19: 類似度≈0.93）。

- NB80: 10K embeddingは `passage: ` プレフィックスで生成
- NB115: Voronoiセントロイドも `passage: ` 空間で学習
- **実運用ではクエリは `query: ` プレフィックスで生成** → セントロイド選択に影響する可能性

## 検証項目
1. passage空間のセントロイドに対して query空間のクエリでprobe → Recallへの影響
2. query/passageの平均差分ベクトルによるセントロイド補正の効果
3. 実運用を想定した正しい評価: ドキュメント=passage, クエリ=query

## NB19の知見
- passage:同士の検索（Baseline_Passage）が最高性能
- query:で検索するとGround Truth自体が変わる → 「何が正解か」が異なる

In [1]:
import sys
import numpy as np
import time
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import gc, torch
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 200
TOP_K = 10
print(f'Configuration: N_QUERIES={N_QUERIES}, TOP_K={TOP_K}')

Configuration: N_QUERIES=200, TOP_K=10


## 1. データ準備: passage embedding と query embedding の生成

同じテキストに対して `passage:` と `query:` の両方でembeddingを生成する。

In [2]:
# passage: embeddingは既存データを使用（NB80で生成済み）
emb_passage_en = np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy')
emb_passage_ja = np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy')
print(f'Passage EN: {emb_passage_en.shape}')
print(f'Passage JA: {emb_passage_ja.shape}')

# query: embeddingを生成（同じテキストにquery:プレフィックスを付けて再生成）
# NB80のテキストを再取得する必要がある
# → Wikipedia 10Kデータを再取得
from datasets import load_dataset

def load_wiki_texts(lang_code, n_docs=10000):
    """NB80と同じ方法でWikipediaテキストを取得"""
    wiki = load_dataset(
        "wikimedia/wikipedia", f"20231101.{lang_code}",
        split="train", streaming=True
    )
    texts = []
    for item in wiki:
        if len(texts) >= n_docs:
            break
        text = item['text'][:500]  # NB80と同じく先頭500文字
        if len(text.strip()) > 50:
            texts.append(text)
    return texts

QUERY_EN_PATH = DATA_DIR / '116_e5_base_en_query_embeddings.npy'
QUERY_JA_PATH = DATA_DIR / '116_e5_base_ja_query_embeddings.npy'

if QUERY_EN_PATH.exists() and QUERY_JA_PATH.exists():
    print('Query embeddings cached, loading...')
    emb_query_en = np.load(QUERY_EN_PATH)
    emb_query_ja = np.load(QUERY_JA_PATH)
else:
    print('Loading Wikipedia texts...')
    texts_en = load_wiki_texts('en', 10000)
    texts_ja = load_wiki_texts('ja', 10000)
    print(f'  EN: {len(texts_en)} texts, JA: {len(texts_ja)} texts')
    
    print('Loading E5-base model...')
    model = SentenceTransformer('intfloat/multilingual-e5-base', device='cuda')
    
    # query: プレフィックスで生成
    print('Encoding with query: prefix...')
    emb_query_en = model.encode(
        [f'query: {t}' for t in texts_en[:len(emb_passage_en)]],
        batch_size=64, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)
    
    emb_query_ja = model.encode(
        [f'query: {t}' for t in texts_ja[:len(emb_passage_ja)]],
        batch_size=64, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)
    
    np.save(QUERY_EN_PATH, emb_query_en)
    np.save(QUERY_JA_PATH, emb_query_ja)
    print(f'Saved: {QUERY_EN_PATH}, {QUERY_JA_PATH}')
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f'Query EN: {emb_query_en.shape}')
print(f'Query JA: {emb_query_ja.shape}')

Passage EN: (10000, 768)
Passage JA: (9990, 768)
Loading Wikipedia texts...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

  EN: 10000 texts, JA: 10000 texts
Loading E5-base model...


Encoding with query: prefix...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Saved: ../data/116_e5_base_en_query_embeddings.npy, ../data/116_e5_base_ja_query_embeddings.npy
Query EN: (10000, 768)
Query JA: (9990, 768)


## 2. passage空間とquery空間の差の分析

In [3]:
# 同一テキストの passage vs query の類似度
n_check = min(len(emb_passage_en), len(emb_query_en))
pair_sims_en = np.sum(emb_passage_en[:n_check] * emb_query_en[:n_check], axis=1)
n_check_ja = min(len(emb_passage_ja), len(emb_query_ja))
pair_sims_ja = np.sum(emb_passage_ja[:n_check_ja] * emb_query_ja[:n_check_ja], axis=1)

print('=== 同一テキストの passage vs query 類似度 ===')
print(f'  EN: mean={pair_sims_en.mean():.4f}, std={pair_sims_en.std():.4f}, '
      f'min={pair_sims_en.min():.4f}, max={pair_sims_en.max():.4f}')
print(f'  JA: mean={pair_sims_ja.mean():.4f}, std={pair_sims_ja.std():.4f}, '
      f'min={pair_sims_ja.min():.4f}, max={pair_sims_ja.max():.4f}')

# 平均差分ベクトル
diff_en = emb_query_en[:n_check] - emb_passage_en[:n_check]
diff_ja = emb_query_ja[:n_check_ja] - emb_passage_ja[:n_check_ja]
mean_diff_en = diff_en.mean(axis=0)
mean_diff_ja = diff_ja.mean(axis=0)

print(f'\n=== 平均差分ベクトル (query - passage) ===')
print(f'  EN: norm={np.linalg.norm(mean_diff_en):.4f}')
print(f'  JA: norm={np.linalg.norm(mean_diff_ja):.4f}')

# 差分ベクトルの一貫性（個別差分と平均差分のcosine）
individual_cos_en = cosine_similarity(diff_en, mean_diff_en.reshape(1, -1)).flatten()
individual_cos_ja = cosine_similarity(diff_ja, mean_diff_ja.reshape(1, -1)).flatten()
print(f'  EN 差分一貫性: mean={individual_cos_en.mean():.4f}, std={individual_cos_en.std():.4f}')
print(f'  JA 差分一貫性: mean={individual_cos_ja.mean():.4f}, std={individual_cos_ja.std():.4f}')
print(f'  → 1.0に近いほど一様なシフト（セントロイド補正が有効）')

=== 同一テキストの passage vs query 類似度 ===
  EN: mean=0.9579, std=0.0104, min=0.8922, max=0.9832
  JA: mean=0.9644, std=0.0105, min=0.8973, max=0.9844

=== 平均差分ベクトル (query - passage) ===
  EN: norm=0.2123
  JA: norm=0.1999
  EN 差分一貫性: mean=0.7397, std=0.0720
  JA 差分一貫性: mean=0.7588, std=0.0646
  → 1.0に近いほど一様なシフト（セントロイド補正が有効）


## 3. Voronoi検索: 4パターン比較

| パターン | ドキュメント | クエリ | セントロイド | 実運用 |
|---------|------------|--------|------------|--------|
| A: passage-passage | passage: | passage: | passage空間 | NB111-114と同じ |
| B: query-passage | passage: | **query:** | passage空間 | **実運用想定** |
| C: 補正セントロイド | passage: | **query:** | passage+差分補正 | 改良案 |
| D: passage-passage (GT=query) | passage: | passage: | passage空間 | GTをquery基準に |

In [4]:
def precompute_ground_truth(doc_embeddings, query_embeddings, query_indices, top_k=10):
    """ground truthを事前計算。doc/queryが異なるembeddingの場合に対応"""
    gt_dict = {}
    cos_all = cosine_similarity(query_embeddings[query_indices], doc_embeddings)
    for i, qi in enumerate(query_indices):
        cos_all[i, qi] = -1  # 自分自身を除外
        gt_dict[qi] = set(np.argsort(cos_all[i])[-top_k:])
    return gt_dict


def build_voronoi(embeddings, n_clusters=256):
    """k-meansでVoronoi分割を構築"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=2048, n_init=3)
    kmeans.fit(emb_normed)
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    return centroids_normed


def build_multi_assign_partitions(doc_embeddings_normed, centroids, n_assign):
    """マルチアサインのパーティションを構築"""
    all_sims = doc_embeddings_normed @ centroids.T
    N, C = all_sims.shape
    partitions = {c: [] for c in range(C)}
    for i in range(N):
        top_pivots = np.argsort(-all_sims[i])[:n_assign]
        for pid in top_pivots:
            partitions[pid].append(i)
    for c in range(C):
        partitions[c] = np.array(partitions[c], dtype=int)
    return partitions


def evaluate_voronoi_prefix(doc_embeddings, query_embeddings_normed, centroids,
                            partitions, gt_dict, query_indices, n_probes, top_k=10):
    """プレフィックス対応のVoronoi検索評価"""
    recalls = []
    candidate_counts = []
    
    for qi in query_indices:
        gt = gt_dict[qi]
        # クエリはquery空間のembeddingでセントロイドとの類似度を計算
        sims = centroids @ query_embeddings_normed[qi]
        top_c = np.argsort(-sims)[:n_probes]
        
        candidates_set = set()
        for c in top_c:
            candidates_set.update(partitions[c].tolist())
        candidates_set.discard(qi)
        candidates = np.array(list(candidates_set))
        candidate_counts.append(len(candidates))
        
        if len(candidates) > 0:
            # rerankもquery embeddingで計算
            cand_sims = cosine_similarity(
                query_embeddings_normed[qi:qi+1], 
                doc_embeddings[candidates] / np.linalg.norm(doc_embeddings[candidates], axis=1, keepdims=True)
            )[0]
            top_in_cand = candidates[np.argsort(-cand_sims)[:top_k]]
            recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            recalls.append(0.0)
    
    return np.mean(recalls), np.mean(candidate_counts)


print('評価関数定義完了')

評価関数定義完了


In [5]:
# セントロイド構築（passage空間）
centroids_passage = build_voronoi(np.vstack([emb_passage_en, emb_passage_ja]), n_clusters=256)
print(f'Centroids (passage空間): {centroids_passage.shape}')

# 補正セントロイド: passage空間セントロイドに平均差分を加えてquery空間に近づける
mean_diff_mixed = np.mean(np.vstack([diff_en, diff_ja]), axis=0)
centroids_corrected = centroids_passage + mean_diff_mixed
centroids_corrected = centroids_corrected / np.linalg.norm(centroids_corrected, axis=1, keepdims=True)
print(f'Centroids (差分補正): {centroids_corrected.shape}')

# 補正前後のセントロイドの類似度
corr_sims = np.sum(centroids_passage * centroids_corrected, axis=1)
print(f'補正前後のセントロイド類似度: mean={corr_sims.mean():.4f}, std={corr_sims.std():.4f}')

# パーティション構築（ドキュメントはpassage空間で割り当て）
probe_range = [2, 3, 4, 5, 8, 10]
assign_range = [1, 2]

rng = np.random.default_rng(42)

# 4パターンの評価を実行
for ds_name, emb_p, emb_q in [('EN', emb_passage_en, emb_query_en), 
                                ('JA', emb_passage_ja, emb_query_ja)]:
    N = min(len(emb_p), len(emb_q))
    emb_p = emb_p[:N]
    emb_q = emb_q[:N]
    
    emb_p_normed = emb_p / np.linalg.norm(emb_p, axis=1, keepdims=True)
    emb_q_normed = emb_q / np.linalg.norm(emb_q, axis=1, keepdims=True)
    
    qi = rng.choice(N, min(N_QUERIES, N // 2), replace=False)
    
    # Ground truth: 2パターン
    gt_pp = precompute_ground_truth(emb_p, emb_p, qi)  # passage→passage
    gt_qp = precompute_ground_truth(emb_p, emb_q, qi)  # query→passage (実運用)
    
    # GT一致率
    overlaps = [len(gt_pp[q] & gt_qp[q]) / TOP_K for q in qi]
    print(f'\n{"="*80}')
    print(f'{ds_name}: GT一致率 (passage vs query基準)')
    print(f'  mean={np.mean(overlaps):.3f}, std={np.std(overlaps):.3f}')
    print(f'{"="*80}')
    
    for n_assign in assign_range:
        partitions = build_multi_assign_partitions(emb_p_normed, centroids_passage, n_assign)
        partitions_corr = build_multi_assign_partitions(emb_p_normed, centroids_passage, n_assign)
        # 注: パーティション割り当てはpassage空間で行う（ドキュメント側）
        
        print(f'\n--- assign={n_assign} ---')
        print(f'{"Pattern":<30} {"P":>3} {"R@10":>8} {"Cands":>7} {"Cand%":>7}')
        print('-' * 58)
        
        for n_p in probe_range:
            # A: passage-passage (NB111-114と同じ)
            r_a, c_a = evaluate_voronoi_prefix(
                emb_p, emb_p_normed, centroids_passage, partitions, gt_pp, qi, n_p)
            
            # B: query→passage空間セントロイド (実運用想定)
            r_b, c_b = evaluate_voronoi_prefix(
                emb_p, emb_q_normed, centroids_passage, partitions, gt_qp, qi, n_p)
            
            # C: query→補正セントロイド
            r_c, c_c = evaluate_voronoi_prefix(
                emb_p, emb_q_normed, centroids_corrected, partitions, gt_qp, qi, n_p)
            
            print(f'{"A: pass→pass":<30} {n_p:>3} {r_a*100:>7.1f}% {c_a:>6.0f} {c_a/N*100:>6.1f}%')
            print(f'{"B: query→pass centroid":<30} {n_p:>3} {r_b*100:>7.1f}% {c_b:>6.0f} {c_b/N*100:>6.1f}%')
            print(f'{"C: query→corrected centroid":<30} {n_p:>3} {r_c*100:>7.1f}% {c_c:>6.0f} {c_c/N*100:>6.1f}%')
            print()

Centroids (passage空間): (256, 768)
Centroids (差分補正): (256, 768)
補正前後のセントロイド類似度: mean=0.9845, std=0.0001



EN: GT一致率 (passage vs query基準)
  mean=0.745, std=0.133

--- assign=1 ---
Pattern                          P     R@10   Cands   Cand%
----------------------------------------------------------


A: pass→pass                     2    70.3%    258    2.6%
B: query→pass centroid           2    67.2%    257    2.6%
C: query→corrected centroid      2    68.5%    252    2.5%



A: pass→pass                     3    76.2%    380    3.8%
B: query→pass centroid           3    73.3%    389    3.9%
C: query→corrected centroid      3    73.9%    382    3.8%



A: pass→pass                     4    80.5%    507    5.1%
B: query→pass centroid           4    77.5%    515    5.1%
C: query→corrected centroid      4    77.7%    506    5.1%



...

A: pass→pass                     3    94.4%    623    6.2%
B: query→pass centroid           3    90.7%    642    6.4%
C: query→corrected centroid      3    90.7%    630    6.3%



A: pass→pass                     4    96.0%    798    8.0%
B: query→pass centroid           4    92.1%    837    8.4%
C: query→corrected centroid      4    92.3%    831    8.3%



A: pass→pass                     5    97.0%    964    9.6%
B: query→pass centroid           5    93.8%   1014   10.2%
C: query→corrected centroid      5    93.6%    999   10.0%



A: pass→pass                     8    98.6%   1450   14.5%
B: query→pass centroid           8    95.4%   1554   15.6%
C: query→corrected centroid      8    95.4%   1537   15.4%



A: pass→pass                    10    99.0%   1783   17.8%
B: query→pass centroid          10    96.4%   1892   18.9%
C: query→corrected centroid     10    96.2%   1882   18.8%



## 4. 評価・考察

### passage vs query の空間差

| 指標 | EN | JA |
|------|-----|-----|
| 同一テキスト passage-query cos | 0.958 | 0.964 |
| 平均差分ベクトル norm | 0.212 | 0.200 |
| 差分一貫性（cosine） | 0.740 | 0.759 |

差分一貫性が0.74-0.76と**完全な一様シフトではない**（1.0なら一様）。テキストごとにquery/passage間のズレ方向が異なるため、単純な平均差分補正の効果は限定的。

### Ground Truthの一致率

| | EN | JA |
|---|-----|-----|
| GT一致率 (passage基準 vs query基準) | **74.5%** | **71.8%** |

passage空間で定義した「正解top-10」とquery空間で定義した「正解top-10」は**約25-28%が異なる**。これはプレフィックスの影響が無視できないことを示す。

### パターンA vs B: プレフィックス不一致のRecall低下

C=256, assign=2での比較:

| P | EN: A(pass) | EN: B(query) | EN低下 | JA: A(pass) | JA: B(query) | JA低下 |
|---|------------|-------------|--------|------------|-------------|--------|
| 2 | 82.2% | 80.3% | -1.9pp | 90.7% | 86.1% | **-4.6pp** |
| 3 | 86.6% | 84.8% | -1.8pp | 94.4% | 90.7% | **-3.7pp** |
| 5 | 91.2% | 89.7% | -1.5pp | 97.0% | 93.8% | **-3.2pp** |
| 10 | 96.2% | 96.0% | -0.2pp | 99.0% | 96.4% | **-2.6pp** |

- **ENでは1-2ppの低下**で影響は軽微
- **JAでは3-5ppの低下**でやや大きいが、壊滅的ではない
- probeを増やせば差は縮まる（P=10ではEN 0.2pp, JA 2.6pp）

### パターンC: セントロイド差分補正の効果

補正セントロイド（passage + 平均差分）の効果は**ほぼゼロ**:
- EN: BとCの差は±0.3pp以内
- JA: BとCの差は±0.3pp以内

差分一貫性が0.74-0.76と低いため、平均差分での一様補正は近傍選択の改善に寄与しない。

### 実運用への影響と対策

**プレフィックス不一致の影響は「probe数を1-2増やせばカバーできる」レベル**:

| 目標 | EN: A(passage) | EN: B(query,実運用) | probe増分 |
|------|---------------|-------------------|----------|
| R@10≥90% | A=2, P=5 | A=2, P=8 | +3 |
| R@10≥85% | A=2, P=3 | A=2, P=4 | +1 |
| R@10≥80% | A=2, P=2 | A=2, P=3 | +1 |

| 目標 | JA: A(passage) | JA: B(query,実運用) | probe増分 |
|------|---------------|-------------------|----------|
| R@10≥90% | A=2, P=3 | A=2, P=5 | +2 |
| R@10≥85% | A=2, P=2 | A=2, P=2 | ±0 |

### 結論

1. **query/passageプレフィックスの影響はEN 1-2pp、JA 3-5pp**。無視はできないが壊滅的ではない
2. **セントロイド差分補正は効果なし**。差分が一様でないため
3. **実運用ではprobeを1-2増やすだけで対応可能**。NB114の推奨値にP+1〜2すればよい
4. **NB111-114のpassage-passage評価は「上限値」として有効**。実運用では数pp低下する前提で設計すること
5. **GT自体が25-28%異なる**ことが最大の要因。Voronoiのセントロイド選択ではなく、cosine rerankの段階で「何が正解か」が変わっている